In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import logging
import platform
import time
from datetime import datetime

import pandas as pd
import numpy as np

# [FIX 1] Ép trực tiếp đường dẫn data vào môi trường của Jupyter để không bị nhận nhầm
os.environ["FRUITNET_DATA_ROOT"] = "/mnt/f/fruitnet-chili-yield-data"

# ============================================================
# 0. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name != "fruitnet-chili-yield":
    if PROJECT_ROOT.parent.name == "fruitnet-chili-yield":
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

assert PROJECT_ROOT.name == "fruitnet-chili-yield", (
    f"Please run this script from fruitnet-chili-yield root directory. Current: {PROJECT_ROOT}"
)

os.chdir(PROJECT_ROOT)


# ============================================================
# 1. WSL / WINDOWS F DRIVE DATA + OUTPUT PATHS
# ============================================================

def get_default_data_root() -> Path:
    """
    Dataset root.
    """
    env_path = os.environ.get("FRUITNET_DATA_ROOT", "").strip()
    if env_path:
        return Path(env_path).expanduser()

    system_name = platform.system().lower()

    f_drive_wsl = Path("/mnt/f/fruitnet-chili-yield-data")
    if system_name == "linux" and f_drive_wsl.exists():
        return f_drive_wsl

    f_drive_win = Path("F:/fruitnet-chili-yield-data")
    if system_name == "windows" and f_drive_win.exists():
        return f_drive_win

    return PROJECT_ROOT / "data"


def get_default_output_root() -> Path:
    """
    Output root for runs, checkpoints, logs, metrics, and artifacts.
    """
    env_path = os.environ.get("FRUITNET_OUTPUT_ROOT", "").strip()
    if env_path:
        return Path(env_path).expanduser()

    system_name = platform.system().lower()

    if system_name == "linux" and Path("/mnt/f").exists():
        return Path("/mnt/f/fruitnet-chili-yield-outputs")

    if system_name == "windows":
        return Path("F:/fruitnet-chili-yield-outputs")

    return PROJECT_ROOT / "outputs"


DATA_ROOT = get_default_data_root().resolve()
OUTPUT_ROOT = get_default_output_root().resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"

MINNE_PROC = PROCESSED_ROOT / "minneapple_yolo"
GWHD_PROC = PROCESSED_ROOT / "gwhd_yolo"

LOG_DIR = OUTPUT_ROOT / "logs"
CONFIG_DATA_DIR = PROJECT_ROOT / "configs" / "data"

RUNS_BENCHMARK_DIR = OUTPUT_ROOT / "runs" / "detectors" / "public_benchmark"
RUNS_VAL_DIR = OUTPUT_ROOT / "runs" / "detectors" / "public_benchmark_val"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "pretrained"
PROJECT_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pretrained"
RESULT_DIR = OUTPUT_ROOT / "results" / "exp01_domain_benchmark"
MARKER_DIR = RESULT_DIR / "job_markers"
METRICS_DIR = RESULT_DIR / "metrics_snapshots"

for d in [
    LOG_DIR,
    CONFIG_DATA_DIR,
    RUNS_BENCHMARK_DIR,
    RUNS_VAL_DIR,
    ARTIFACT_DIR,
    RESULT_DIR,
    MARKER_DIR,
    METRICS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. SAFE WRITE HELPERS
# ============================================================

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding=encoding)
    os.replace(tmp, path)


def atomic_write_json(path: Path, data: dict) -> None:
    atomic_write_text(path, json.dumps(data, indent=2, ensure_ascii=False) + "\n")


def atomic_write_df(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def append_jsonl_fsync(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(payload, ensure_ascii=False) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)
        f.flush()
        os.fsync(f.fileno())


# ============================================================
# 3. LOGGING (ĐÃ FIX CHO JUPYTER NOTEBOOK)
# ============================================================

LOG_PATH = LOG_DIR / f"03_public_benchmark_table41_resume_safe_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()

# Lấy đúng luồng stdout/stderr hiện tại của Jupyter Notebook thay vì của hệ thống ngầm
jupyter_stdout = sys.stdout
jupyter_stderr = sys.stderr

_log_fp = open(LOG_PATH, "a", encoding="utf-8")
sys.stdout = Tee(jupyter_stdout, _log_fp)
sys.stderr = Tee(jupyter_stderr, _log_fp)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(jupyter_stdout), # Trỏ logger về luồng của Jupyter
    ],
)

logging.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logging.info("DATA_ROOT=%s", DATA_ROOT)
logging.info("OUTPUT_ROOT=%s", OUTPUT_ROOT)
logging.info("MINNE_PROC=%s", MINNE_PROC)
logging.info("GWHD_PROC=%s", GWHD_PROC)
logging.info("RUNS_BENCHMARK_DIR=%s", RUNS_BENCHMARK_DIR)
logging.info("RUNS_VAL_DIR=%s", RUNS_VAL_DIR)
logging.info("RESULT_DIR=%s", RESULT_DIR)
logging.info("Python=%s", sys.version)
logging.info("Platform=%s", platform.platform())
logging.info("Log file=%s", LOG_PATH)


# ============================================================
# 4. WRITE DATA YAML FILES WITH CURRENT PATHS
# ============================================================

MINNE_YAML = CONFIG_DATA_DIR / "minneapple.yaml"
GWHD_YAML = CONFIG_DATA_DIR / "gwhd.yaml"


def write_dataset_yamls() -> None:
    atomic_write_text(
        MINNE_YAML,
        f"""
path: {MINNE_PROC.as_posix()}
train: images/train
val: images/val
test: images/val
names:
  0: apple
""".strip() + "\n",
    )

    atomic_write_text(
        GWHD_YAML,
        f"""
path: {GWHD_PROC.as_posix()}
train: images/train
val: images/val
test: images/val
names:
  0: wheat_head
""".strip() + "\n",
    )

    logging.info("Wrote dataset YAML files: %s, %s", MINNE_YAML, GWHD_YAML)


write_dataset_yamls()


DATASETS = {
    "MinneApple": {
        "yaml": MINNE_YAML,
        "root": MINNE_PROC,
        "train_images": MINNE_PROC / "images" / "train",
        "val_images": MINNE_PROC / "images" / "val",
        "train_labels": MINNE_PROC / "labels" / "train",
        "val_labels": MINNE_PROC / "labels" / "val",
    },
    "GWHD": {
        "yaml": GWHD_YAML,
        "root": GWHD_PROC,
        "train_images": GWHD_PROC / "images" / "train",
        "val_images": GWHD_PROC / "images" / "val",
        "train_labels": GWHD_PROC / "labels" / "train",
        "val_labels": GWHD_PROC / "labels" / "val",
    },
}


def check_dataset_ready(dataset_name: str, spec: dict) -> bool:
    missing = []
    for key in ["root", "train_images", "val_images", "train_labels", "val_labels"]:
        p = spec[key]
        if not p.exists():
            missing.append(str(p))

    if missing:
        logging.warning("%s dataset missing. Missing paths:\n%s", dataset_name, "\n".join(missing))
        print(f"\nWARNING: {dataset_name} dataset is missing. It will be skipped.")
        for m in missing:
            print("  -", m)
        return False

    n_train_img = len(list(spec["train_images"].glob("*")))
    n_val_img = len(list(spec["val_images"].glob("*")))
    n_train_lab = len(list(spec["train_labels"].glob("*.txt")))
    n_val_lab = len(list(spec["val_labels"].glob("*.txt")))

    logging.info(
        "%s ready: train_images=%s val_images=%s train_labels=%s val_labels=%s",
        dataset_name,
        n_train_img,
        n_val_img,
        n_train_lab,
        n_val_lab,
    )

    print(f"\n{dataset_name} dataset ready:")
    print("  root:", spec["root"])
    print("  yaml:", spec["yaml"])
    print("  train images:", n_train_img)
    print("  val images:", n_val_img)
    print("  train labels:", n_train_lab)
    print("  val labels:", n_val_lab)

    if n_train_img == 0 or n_val_img == 0:
        logging.warning("%s has empty image folders. It will be skipped.", dataset_name)
        return False

    return True


VALID_DATASETS = {}
for dataset_name, spec in DATASETS.items():
    if check_dataset_ready(dataset_name, spec):
        VALID_DATASETS[dataset_name] = spec

if not VALID_DATASETS:
    error_msg = (
        "\nNo benchmark dataset is ready.\n\n"
        f"Expected dataset root: {DATA_ROOT}\n\n"
        "Run dataset preparation first:\n"
        "  python 01_download_prepare_datasets_wsl_f_drive.py\n\n"
        "Or set correct path using Environment Variable:\n"
        "  export FRUITNET_DATA_ROOT=/mnt/f/fruitnet-chili-yield-data\n"
    )
    logging.error(error_msg)
    # Dùng raise thay cho sys.exit để không làm crash Jupyter Kernel
    raise RuntimeError(error_msg)


# ============================================================
# 5. BENCHMARK CONFIG
# ============================================================

IMG_SIZE = int(os.environ.get("IMG_SIZE", 640))
BATCH = int(os.environ.get("BATCH", 8))
EPOCHS = int(os.environ.get("EPOCHS", 100))
WORKERS = int(os.environ.get("WORKERS", 8))
SEED = int(os.environ.get("SEED", 42))
PATIENCE = int(os.environ.get("PATIENCE", 30))
SAVE_PERIOD = int(os.environ.get("SAVE_PERIOD", 1))

RUN_TRAINING = os.environ.get("RUN_TRAINING", "1").strip() not in ["0", "false", "False", "no", "NO"]
RUN_VALIDATION = os.environ.get("RUN_VALIDATION", "1").strip() not in ["0", "false", "False", "no", "NO"]
FORCE_RETRAIN_DONE = os.environ.get("FORCE_RETRAIN_DONE", "0").strip() in ["1", "true", "True", "yes", "YES"]
FORCE_REVALIDATE_DONE = os.environ.get("FORCE_REVALIDATE_DONE", "0").strip() in ["1", "true", "True", "yes", "YES"]

DEVICE_ENV = os.environ.get("DEVICE", "").strip()

MODEL_SOURCES = {
    "YOLOv5s": {
        "model_key": "yolov5s",
        "artifact_name": "yolov5s_coco_best.pt",
        "fallback": "yolov5su.pt",
    },
    "YOLOv8n": {
        "model_key": "yolov8n",
        "artifact_name": "yolov8n_coco_best.pt",
        "fallback": "yolov8n.pt",
    },
    "FruitNet B": {
        "model_key": "fruitnet_b",
        "artifact_name": "fruitnet_b_coco_best.pt",
        "fallback": "configs/models/fruitnet_b.yaml",
    },
    "FruitNet GCS": {
        "model_key": "fruitnet_gcs",
        "artifact_name": "fruitnet_gcs_coco_best.pt",
        "fallback": "configs/models/fruitnet_gcs.yaml",
    },
}


# ============================================================
# 6. IMPORT TORCH / ULTRALYTICS / FRUITNET MODULES
# ============================================================

import torch
from ultralytics import YOLO

try:
    from models.fruitnet.register_ultralytics_modules import register_fruitnet_modules

    register_fruitnet_modules()
    logging.info("FruitNet custom Ultralytics modules registered.")
    print("\nFruitNet custom modules registered.")
except Exception as e:
    logging.warning("Could not register FruitNet custom modules: %s", repr(e))
    print("\nWarning: Could not register FruitNet custom modules.")
    print("If you benchmark FruitNet models, make sure this file exists:")
    print("  models/fruitnet/register_ultralytics_modules.py")
    print("Error:", repr(e))

logging.info("torch=%s cuda=%s", torch.__version__, torch.cuda.is_available())

if DEVICE_ENV:
    DEVICE = DEVICE_ENV
else:
    DEVICE = 0 if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    logging.info("GPU=%s", torch.cuda.get_device_name(0))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Training will use CPU unless DEVICE is changed.")

print("\nBenchmark datasets:")
for name, spec in VALID_DATASETS.items():
    print(f"  {name}: {spec['yaml'].resolve()}")

print("\nSettings:")
print("  PROJECT_ROOT:", PROJECT_ROOT)
print("  DATA_ROOT:", DATA_ROOT)
print("  OUTPUT_ROOT:", OUTPUT_ROOT)
print("  RUNS_BENCHMARK_DIR:", RUNS_BENCHMARK_DIR)
print("  RUNS_VAL_DIR:", RUNS_VAL_DIR)
print("  RESULT_DIR:", RESULT_DIR)
print("  LOG_PATH:", LOG_PATH)
print("  EPOCHS:", EPOCHS)
print("  BATCH:", BATCH)
print("  IMG_SIZE:", IMG_SIZE)
print("  WORKERS:", WORKERS)
print("  PATIENCE:", PATIENCE)
print("  SAVE_PERIOD:", SAVE_PERIOD)
print("  RUN_TRAINING:", RUN_TRAINING)
print("  RUN_VALIDATION:", RUN_VALIDATION)
print("  FORCE_RETRAIN_DONE:", FORCE_RETRAIN_DONE)
print("  FORCE_REVALIDATE_DONE:", FORCE_REVALIDATE_DONE)
print("  DEVICE:", DEVICE)


# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

PROGRESS_PATH = RESULT_DIR / "table_4_1_progress.csv"
FULL_CSV_PATH = RESULT_DIR / "table_4_1_public_benchmarks.csv"
MANUSCRIPT_CSV_PATH = RESULT_DIR / "table_4_1_manuscript_ready.csv"
STATUS_PATH = RESULT_DIR / "table_4_1_status.json"


def clean_name(s: str) -> str:
    return s.lower().replace(" ", "_").replace("-", "_").replace("/", "_")


def job_id_for(dataset_name: str, model_name: str) -> str:
    return f"{clean_name(dataset_name)}_{clean_name(model_name)}"


def resolve_artifact_path(artifact_name: str) -> Path | None:
    candidates = [
        ARTIFACT_DIR / artifact_name,
        PROJECT_ARTIFACT_DIR / artifact_name,
        OUTPUT_ROOT / "artifacts" / "pretrained" / artifact_name,
    ]

    for p in candidates:
        if p.exists():
            return p.resolve()

    return None


def choose_model_source(spec: dict) -> str:
    artifact_path = resolve_artifact_path(spec["artifact_name"])
    if artifact_path is not None:
        return str(artifact_path)
    return spec["fallback"]


def model_uri_exists_or_builtin(model_uri: str) -> bool:
    lower = model_uri.lower()
    if lower.endswith(".pt"):
        return True
    return Path(model_uri).exists()


def should_use_pretrained(model_uri: str) -> bool:
    return model_uri.lower().endswith(".pt")


def is_finished_resume_error(e: Exception) -> bool:
    s = repr(e).lower()
    return "finished" in s and "nothing to resume" in s


def metrics_to_dict(metrics):
    box = getattr(metrics, "box", None)

    out = {
        "precision": np.nan,
        "recall": np.nan,
        "map50": np.nan,
        "map50_95": np.nan,
        "fitness": np.nan,
    }

    if box is not None:
        for key, attr in [
            ("precision", "mp"),
            ("recall", "mr"),
            ("map50", "map50"),
            ("map50_95", "map"),
        ]:
            try:
                out[key] = float(getattr(box, attr))
            except Exception:
                pass

    try:
        out["fitness"] = float(metrics.fitness)
    except Exception:
        pass

    return out


def find_best_and_last(run_dir: Path):
    weights_dir = run_dir / "weights"
    best = weights_dir / "best.pt"
    last = weights_dir / "last.pt"
    return best if best.exists() else None, last if last.exists() else None


def scalarize(value):
    try:
        if hasattr(value, "item"):
            return float(value.item())
        if isinstance(value, (int, float, str, bool)):
            return value
        return str(value)
    except Exception:
        return str(value)


def collect_trainer_metrics(trainer) -> dict:
    payload = {}
    for attr in ["epoch", "best_fitness", "fitness"]:
        try:
            payload[attr] = scalarize(getattr(trainer, attr))
        except Exception:
            pass

    for attr in ["metrics", "loss_items", "tloss"]:
        try:
            obj = getattr(trainer, attr)
            if isinstance(obj, dict):
                payload[attr] = {str(k): scalarize(v) for k, v in obj.items()}
            else:
                payload[attr] = scalarize(obj)
        except Exception:
            pass

    return payload


def add_crash_safe_callbacks(model, context: dict) -> None:
    def callback(event_name: str):
        def _cb(trainer):
            payload = {
                "time": now_iso(),
                "event": event_name,
                "job_id": context["job_id"],
                "dataset": context["dataset"],
                "model": context["model"],
                "run_dir": str(context["run_dir"]),
                "last_pt": str(context["last_pt"]),
                "best_pt": str(context["best_pt"]),
            }
            payload.update(collect_trainer_metrics(trainer))
            append_jsonl_fsync(context["events_jsonl"], payload)
            atomic_write_json(context["heartbeat_json"], payload)
        return _cb

    for event_name in ["on_train_epoch_end", "on_fit_epoch_end", "on_model_save", "on_train_end"]:
        try:
            model.add_callback(event_name, callback(event_name))
        except Exception as e:
            logging.warning("Could not add callback %s for %s: %s", event_name, context["job_id"], repr(e))


def snapshot_training_metrics(job_id: str, run_dir: Path) -> dict:
    snapshot = {
        "job_id": job_id,
        "run_dir": str(run_dir),
        "results_csv": "",
        "args_yaml": "",
        "copied_files": [],
    }

    dst_dir = METRICS_DIR / job_id / "train"
    dst_dir.mkdir(parents=True, exist_ok=True)

    for fname in ["results.csv", "args.yaml"]:
        src = run_dir / fname
        if src.exists():
            dst = dst_dir / fname
            shutil.copy2(src, dst)
            snapshot[fname.replace(".", "_")] = str(dst)
            snapshot["copied_files"].append(str(dst))

    for p in run_dir.glob("*.png"):
        dst = dst_dir / p.name
        shutil.copy2(p, dst)
        snapshot["copied_files"].append(str(dst))

    return snapshot


def snapshot_validation_metrics(job_id: str, val_run_dir: Path) -> dict:
    snapshot = {
        "job_id": job_id,
        "val_run_dir": str(val_run_dir),
        "copied_files": [],
    }

    dst_dir = METRICS_DIR / job_id / "val"
    dst_dir.mkdir(parents=True, exist_ok=True)

    for fname in ["results.csv", "args.yaml"]:
        src = val_run_dir / fname
        if src.exists():
            dst = dst_dir / fname
            shutil.copy2(src, dst)
            snapshot[fname.replace(".", "_")] = str(dst)
            snapshot["copied_files"].append(str(dst))

    for p in val_run_dir.glob("*.png"):
        dst = dst_dir / p.name
        shutil.copy2(p, dst)
        snapshot["copied_files"].append(str(dst))

    return snapshot


def load_existing_progress() -> dict:
    if not PROGRESS_PATH.exists():
        return {}
    try:
        df = pd.read_csv(PROGRESS_PATH)
        if "job_id" not in df.columns:
            return {}
        return {str(r["job_id"]): r.to_dict() for _, r in df.iterrows()}
    except Exception as e:
        logging.warning("Could not read existing progress CSV: %s", repr(e))
        return {}


def build_initial_rows() -> list[dict]:
    existing = load_existing_progress()
    rows = []

    for dataset_name, dataset_spec in VALID_DATASETS.items():
        for model_name, model_spec in MODEL_SOURCES.items():
            jid = job_id_for(dataset_name, model_name)
            run_dir = RUNS_BENCHMARK_DIR / jid
            val_run_dir = RUNS_VAL_DIR / jid
            best, last = find_best_and_last(run_dir)

            train_done_marker = MARKER_DIR / f"{jid}.train.done.json"
            val_done_marker = MARKER_DIR / f"{jid}.val.done.json"
            heartbeat = MARKER_DIR / f"{jid}.heartbeat.json"
            events_jsonl = MARKER_DIR / f"{jid}.events.jsonl"
            val_metrics_json = METRICS_DIR / jid / "val_metrics.json"

            row = {
                "job_id": jid,
                "dataset": dataset_name,
                "dataset_yaml": str(dataset_spec["yaml"].resolve()),
                "dataset_root": str(dataset_spec["root"].resolve()),
                "model": model_name,
                "model_key": model_spec["model_key"],
                "model_uri_used": choose_model_source(model_spec),
                "run_dir": str(run_dir),
                "val_run_dir": str(val_run_dir),
                "status": "pending",
                "train_status": "pending",
                "val_status": "pending",
                "best_pt": str(best) if best else "",
                "last_pt": str(last) if last else "",
                "resume_from": "",
                "precision": np.nan,
                "recall": np.nan,
                "mAP@0.5": np.nan,
                "mAP@0.5:0.95": np.nan,
                "fitness": np.nan,
                "epochs": EPOCHS,
                "batch": BATCH,
                "imgsz": IMG_SIZE,
                "workers": WORKERS,
                "device": DEVICE,
                "seed": SEED,
                "patience": PATIENCE,
                "save_period": SAVE_PERIOD,
                "train_done_marker": str(train_done_marker),
                "val_done_marker": str(val_done_marker),
                "heartbeat_json": str(heartbeat),
                "events_jsonl": str(events_jsonl),
                "val_metrics_json": str(val_metrics_json),
                "started_at": "",
                "finished_at": "",
                "error": "",
            }

            if jid in existing:
                old = existing[jid]
                for k, v in old.items():
                    if k in row and pd.notna(v):
                        row[k] = v

                # Refresh live paths in case OUTPUT_ROOT changed.
                row["model_uri_used"] = choose_model_source(model_spec)
                row["run_dir"] = str(run_dir)
                row["val_run_dir"] = str(val_run_dir)
                row["best_pt"] = str(best) if best else row.get("best_pt", "")
                row["last_pt"] = str(last) if last else row.get("last_pt", "")
                row["train_done_marker"] = str(train_done_marker)
                row["val_done_marker"] = str(val_done_marker)
                row["heartbeat_json"] = str(heartbeat)
                row["events_jsonl"] = str(events_jsonl)
                row["val_metrics_json"] = str(val_metrics_json)

            rows.append(row)

    return rows


def save_progress(rows: list[dict]) -> None:
    df = pd.DataFrame(rows)
    atomic_write_df(PROGRESS_PATH, df)
    atomic_write_df(FULL_CSV_PATH, df)

    display_cols = [
        "dataset",
        "model",
        "precision",
        "recall",
        "mAP@0.5",
        "mAP@0.5:0.95",
        "fitness",
        "status",
        "train_status",
        "val_status",
        "best_pt",
        "error",
    ]
    table = df[display_cols].copy()
    for c in ["precision", "recall", "mAP@0.5", "mAP@0.5:0.95", "fitness"]:
        table[c] = pd.to_numeric(table[c], errors="coerce").round(4)

    atomic_write_df(MANUSCRIPT_CSV_PATH, table)


def update_row(rows: list[dict], jid: str, **updates) -> dict:
    for row in rows:
        if row["job_id"] == jid:
            row.update(updates)
            save_progress(rows)
            return row
    raise KeyError(jid)


def train_done_marker_is_valid(row: dict) -> bool:
    if FORCE_RETRAIN_DONE:
        return False
    marker = Path(str(row["train_done_marker"]))
    best = Path(str(row.get("best_pt", ""))) if str(row.get("best_pt", "")).strip() else None
    last = Path(str(row.get("last_pt", ""))) if str(row.get("last_pt", "")).strip() else None
    return marker.exists() and ((best is not None and best.exists()) or (last is not None and last.exists()))


def val_done_marker_is_valid(row: dict) -> bool:
    if FORCE_REVALIDATE_DONE:
        return False
    marker = Path(str(row["val_done_marker"]))
    metrics_path = Path(str(row["val_metrics_json"]))
    return marker.exists() and metrics_path.exists()


def load_saved_val_metrics(row: dict) -> dict:
    metrics_path = Path(str(row["val_metrics_json"]))
    if not metrics_path.exists():
        return {}
    try:
        return json.loads(metrics_path.read_text(encoding="utf-8"))
    except Exception:
        return {}


# ============================================================
# 8. BENCHMARK ONE MODEL ON ONE DATASET WITH AUTO RESUME
# ============================================================

def benchmark_one(dataset_name: str, dataset_spec: dict, model_name: str, model_spec: dict, rows: list[dict]) -> dict:
    jid = job_id_for(dataset_name, model_name)
    row = next(r for r in rows if r["job_id"] == jid)

    dataset_yaml = dataset_spec["yaml"]
    model_uri = choose_model_source(model_spec)
    run_dir = RUNS_BENCHMARK_DIR / jid
    val_run_dir = RUNS_VAL_DIR / jid

    train_done_marker = MARKER_DIR / f"{jid}.train.done.json"
    val_done_marker = MARKER_DIR / f"{jid}.val.done.json"
    heartbeat_json = MARKER_DIR / f"{jid}.heartbeat.json"
    events_jsonl = MARKER_DIR / f"{jid}.events.jsonl"
    val_metrics_json = METRICS_DIR / jid / "val_metrics.json"

    best, last = find_best_and_last(run_dir)

    row.update({
        "dataset_yaml": str(dataset_yaml.resolve()),
        "dataset_root": str(dataset_spec["root"].resolve()),
        "model_uri_used": model_uri,
        "run_dir": str(run_dir),
        "val_run_dir": str(val_run_dir),
        "best_pt": str(best) if best else "",
        "last_pt": str(last) if last else "",
        "train_done_marker": str(train_done_marker),
        "val_done_marker": str(val_done_marker),
        "heartbeat_json": str(heartbeat_json),
        "events_jsonl": str(events_jsonl),
        "val_metrics_json": str(val_metrics_json),
        "error": "",
    })
    save_progress(rows)

    logging.info("Benchmark job: dataset=%s model=%s jid=%s uri=%s", dataset_name, model_name, jid, model_uri)

    if not model_uri_exists_or_builtin(model_uri):
        err = f"Model config not found: {model_uri}"
        logging.error(err)
        update_row(rows, jid, status="failed", train_status="failed", error=err, finished_at=now_iso())
        return row

    context = {
        "job_id": jid,
        "dataset": dataset_name,
        "model": model_name,
        "run_dir": run_dir,
        "best_pt": run_dir / "weights" / "best.pt",
        "last_pt": run_dir / "weights" / "last.pt",
        "heartbeat_json": heartbeat_json,
        "events_jsonl": events_jsonl,
    }

    # ---------------------------
    # 8.1 Training / Resume
    # ---------------------------
    if RUN_TRAINING and not train_done_marker_is_valid(row):
        started_at = now_iso()
        best, last = find_best_and_last(run_dir)
        resume_from = str(last) if last and last.exists() else ""

        update_row(
            rows,
            jid,
            status="running_train_resume" if resume_from else "running_train_new",
            train_status="running_resume" if resume_from else "running_new",
            started_at=started_at,
            resume_from=resume_from,
            error="",
        )

        start_payload = {
            "time": started_at,
            "event": "train_start",
            "job_id": jid,
            "dataset": dataset_name,
            "model": model_name,
            "model_uri": model_uri,
            "run_dir": str(run_dir),
            "resume_from": resume_from,
            "dataset_yaml": str(dataset_yaml.resolve()),
            "output_root": str(OUTPUT_ROOT),
        }
        append_jsonl_fsync(events_jsonl, start_payload)
        atomic_write_json(heartbeat_json, start_payload)

        try:
            if resume_from:
                logging.info("Resuming benchmark job %s from %s", jid, resume_from)
                model = YOLO(resume_from)
                add_crash_safe_callbacks(model, context)
                model.train(resume=True)
            else:
                logging.info("Starting new benchmark training job %s from %s", jid, model_uri)
                model = YOLO(model_uri)
                add_crash_safe_callbacks(model, context)

                model.train(
                    data=str(dataset_yaml),
                    epochs=EPOCHS,
                    imgsz=IMG_SIZE,
                    batch=BATCH,
                    workers=WORKERS,
                    device=DEVICE,
                    seed=SEED,
                    patience=PATIENCE,
                    project=str(RUNS_BENCHMARK_DIR),
                    name=jid,
                    exist_ok=True,
                    verbose=True,
                    pretrained=should_use_pretrained(model_uri),
                    plots=True,
                    save=True,
                    save_period=SAVE_PERIOD,
                )

            best, last = find_best_and_last(run_dir)
            metrics_snapshot = snapshot_training_metrics(jid, run_dir)

            train_done_payload = {
                "time": now_iso(),
                "event": "train_done",
                "job_id": jid,
                "dataset": dataset_name,
                "model": model_name,
                "model_uri": model_uri,
                "run_dir": str(run_dir),
                "best_pt": str(best) if best else "",
                "last_pt": str(last) if last else "",
                "metrics_snapshot": metrics_snapshot,
            }
            atomic_write_json(train_done_marker, train_done_payload)
            append_jsonl_fsync(events_jsonl, train_done_payload)
            atomic_write_json(heartbeat_json, train_done_payload)

            update_row(
                rows,
                jid,
                status="train_ok",
                train_status="ok",
                best_pt=str(best) if best else "",
                last_pt=str(last) if last else "",
                error="",
            )

        except Exception as e:
            best, last = find_best_and_last(run_dir)

            if is_finished_resume_error(e) and (best or last):
                metrics_snapshot = snapshot_training_metrics(jid, run_dir)
                train_done_payload = {
                    "time": now_iso(),
                    "event": "train_already_finished_detected",
                    "job_id": jid,
                    "dataset": dataset_name,
                    "model": model_name,
                    "run_dir": str(run_dir),
                    "best_pt": str(best) if best else "",
                    "last_pt": str(last) if last else "",
                    "metrics_snapshot": metrics_snapshot,
                    "resume_error": repr(e),
                }
                atomic_write_json(train_done_marker, train_done_payload)
                append_jsonl_fsync(events_jsonl, train_done_payload)
                atomic_write_json(heartbeat_json, train_done_payload)

                update_row(
                    rows,
                    jid,
                    status="train_ok_already_finished",
                    train_status="ok_already_finished",
                    best_pt=str(best) if best else "",
                    last_pt=str(last) if last else "",
                    error="",
                )
            else:
                err = repr(e)
                fail_payload = {
                    "time": now_iso(),
                    "event": "train_failed",
                    "job_id": jid,
                    "dataset": dataset_name,
                    "model": model_name,
                    "run_dir": str(run_dir),
                    "best_pt": str(best) if best else "",
                    "last_pt": str(last) if last else "",
                    "error": err,
                }
                append_jsonl_fsync(events_jsonl, fail_payload)
                atomic_write_json(heartbeat_json, fail_payload)

                update_row(
                    rows,
                    jid,
                    status="failed",
                    train_status="failed",
                    best_pt=str(best) if best else "",
                    last_pt=str(last) if last else "",
                    finished_at=now_iso(),
                    error=err,
                )
                logging.exception("Benchmark training failed: %s", jid)
                return next(r for r in rows if r["job_id"] == jid)

    elif RUN_TRAINING and train_done_marker_is_valid(row):
        logging.info("Skipping completed training job %s", jid)
        update_row(rows, jid, status="train_skipped_completed", train_status="skipped_completed")

    elif not RUN_TRAINING:
        logging.info("RUN_TRAINING=False for %s. Validation will use best.pt/last.pt if available, otherwise model source.", jid)
        update_row(rows, jid, status="train_skipped_run_training_false", train_status="skipped_run_training_false")

    # Refresh checkpoint paths after training or skipping.
    best, last = find_best_and_last(run_dir)
    update_row(rows, jid, best_pt=str(best) if best else "", last_pt=str(last) if last else "")

    # ---------------------------
    # 8.2 Validation / Metrics
    # ---------------------------
    row = next(r for r in rows if r["job_id"] == jid)

    if RUN_VALIDATION and val_done_marker_is_valid(row):
        saved = load_saved_val_metrics(row)
        update_row(
            rows,
            jid,
            status="ok",
            val_status="skipped_completed",
            precision=saved.get("precision", row.get("precision", np.nan)),
            recall=saved.get("recall", row.get("recall", np.nan)),
            **{
                "mAP@0.5": saved.get("mAP@0.5", row.get("mAP@0.5", np.nan)),
                "mAP@0.5:0.95": saved.get("mAP@0.5:0.95", row.get("mAP@0.5:0.95", np.nan)),
            },
            fitness=saved.get("fitness", row.get("fitness", np.nan)),
            finished_at=now_iso(),
            error="",
        )
        return next(r for r in rows if r["job_id"] == jid)

    if RUN_VALIDATION:
        update_row(rows, jid, status="running_validation", val_status="running", error="")

        val_model_uri = str(best) if best else str(last) if last else model_uri

        val_start_payload = {
            "time": now_iso(),
            "event": "val_start",
            "job_id": jid,
            "dataset": dataset_name,
            "model": model_name,
            "val_model_uri": val_model_uri,
            "dataset_yaml": str(dataset_yaml.resolve()),
            "val_run_dir": str(val_run_dir),
        }
        append_jsonl_fsync(events_jsonl, val_start_payload)
        atomic_write_json(heartbeat_json, val_start_payload)

        try:
            val_model = YOLO(val_model_uri)

            metrics = val_model.val(
                data=str(dataset_yaml),
                imgsz=IMG_SIZE,
                batch=BATCH,
                device=DEVICE,
                split="val",
                plots=True,
                verbose=True,
                project=str(RUNS_VAL_DIR),
                name=jid,
                exist_ok=True,
            )

            m = metrics_to_dict(metrics)

            val_metrics_payload = {
                "time": now_iso(),
                "event": "val_done",
                "job_id": jid,
                "dataset": dataset_name,
                "model": model_name,
                "val_model_uri": val_model_uri,
                "precision": m["precision"],
                "recall": m["recall"],
                "mAP@0.5": m["map50"],
                "mAP@0.5:0.95": m["map50_95"],
                "fitness": m["fitness"],
                "snapshot": snapshot_validation_metrics(jid, val_run_dir),
            }

            atomic_write_json(val_metrics_json, val_metrics_payload)
            atomic_write_json(val_done_marker, val_metrics_payload)
            append_jsonl_fsync(events_jsonl, val_metrics_payload)
            atomic_write_json(heartbeat_json, val_metrics_payload)

            update_row(
                rows,
                jid,
                status="ok",
                val_status="ok",
                precision=m["precision"],
                recall=m["recall"],
                **{
                    "mAP@0.5": m["map50"],
                    "mAP@0.5:0.95": m["map50_95"],
                },
                fitness=m["fitness"],
                finished_at=now_iso(),
                error="",
            )

        except Exception as e:
            err = repr(e)
            fail_payload = {
                "time": now_iso(),
                "event": "val_failed",
                "job_id": jid,
                "dataset": dataset_name,
                "model": model_name,
                "val_model_uri": val_model_uri,
                "error": err,
            }
            append_jsonl_fsync(events_jsonl, fail_payload)
            atomic_write_json(heartbeat_json, fail_payload)

            update_row(
                rows,
                jid,
                status="failed_validation",
                val_status="failed",
                finished_at=now_iso(),
                error=err,
            )
            logging.exception("Benchmark validation failed: %s", jid)

    else:
        update_row(rows, jid, status="ok_no_validation", val_status="skipped_run_validation_false", finished_at=now_iso())

    return next(r for r in rows if r["job_id"] == jid)


# ============================================================
# 9. RUN FULL BENCHMARK TABLE 4.1
# ============================================================

rows = build_initial_rows()
save_progress(rows)

master_status = {
    "time": now_iso(),
    "event": "script_start",
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "minneapple_proc": str(MINNE_PROC),
    "gwhd_proc": str(GWHD_PROC),
    "minneapple_yaml": str(MINNE_YAML.resolve()),
    "gwhd_yaml": str(GWHD_YAML.resolve()),
    "runs_benchmark_dir": str(RUNS_BENCHMARK_DIR),
    "runs_val_dir": str(RUNS_VAL_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "project_artifact_dir": str(PROJECT_ARTIFACT_DIR),
    "result_dir": str(RESULT_DIR),
    "progress_csv": str(PROGRESS_PATH),
    "full_csv": str(FULL_CSV_PATH),
    "manuscript_csv": str(MANUSCRIPT_CSV_PATH),
    "log_file": str(LOG_PATH),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "run_training": RUN_TRAINING,
    "run_validation": RUN_VALIDATION,
    "force_retrain_done": FORCE_RETRAIN_DONE,
    "force_revalidate_done": FORCE_REVALIDATE_DONE,
    "epochs": EPOCHS,
    "batch": BATCH,
    "imgsz": IMG_SIZE,
    "workers": WORKERS,
    "patience": PATIENCE,
    "save_period": SAVE_PERIOD,
}
atomic_write_json(STATUS_PATH, master_status)

for dataset_name, dataset_spec in VALID_DATASETS.items():
    for model_name, model_spec in MODEL_SOURCES.items():
        row = benchmark_one(dataset_name, dataset_spec, model_name, model_spec, rows)
        print("\nLatest benchmark result:")
        print(pd.DataFrame([row]))
        time.sleep(1)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

save_progress(rows)

df = pd.DataFrame(rows)

display_cols = [
    "dataset",
    "model",
    "precision",
    "recall",
    "mAP@0.5",
    "mAP@0.5:0.95",
    "fitness",
    "status",
    "train_status",
    "val_status",
    "best_pt",
    "error",
]
table = df[display_cols].copy()
for c in ["precision", "recall", "mAP@0.5", "mAP@0.5:0.95", "fitness"]:
    table[c] = pd.to_numeric(table[c], errors="coerce").round(4)

master_status.update(
    {
        "time": now_iso(),
        "event": "script_end",
        "full_csv": str(FULL_CSV_PATH),
        "manuscript_csv": str(MANUSCRIPT_CSV_PATH),
        "progress_csv": str(PROGRESS_PATH),
        "log_file": str(LOG_PATH),
    }
)
atomic_write_json(STATUS_PATH, master_status)

print("\nSaved:")
print(" -", PROGRESS_PATH)
print(" -", FULL_CSV_PATH)
print(" -", MANUSCRIPT_CSV_PATH)
print(" -", STATUS_PATH)
print(" -", LOG_PATH)

print("\nManuscript-ready Table 4.1:")
print(table)

print("\nAll training checkpoints are under:", RUNS_BENCHMARK_DIR)
print("All validation runs are under:", RUNS_VAL_DIR)
print("Metrics snapshots are under:", METRICS_DIR)

2026-06-22 02:55:10,075 | INFO | PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
2026-06-22 02:55:10,077 | INFO | DATA_ROOT=/mnt/f/fruitnet-chili-yield-data
2026-06-22 02:55:10,078 | INFO | OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
2026-06-22 02:55:10,078 | INFO | MINNE_PROC=/mnt/f/fruitnet-chili-yield-data/processed/minneapple_yolo
2026-06-22 02:55:10,079 | INFO | GWHD_PROC=/mnt/f/fruitnet-chili-yield-data/processed/gwhd_yolo
2026-06-22 02:55:10,080 | INFO | RUNS_BENCHMARK_DIR=/mnt/f/fruitnet-chili-yield-outputs/runs/detectors/public_benchmark
2026-06-22 02:55:10,080 | INFO | RUNS_VAL_DIR=/mnt/f/fruitnet-chili-yield-outputs/runs/detectors/public_benchmark_val
2026-06-22 02:55:10,081 | INFO | RESULT_DIR=/mnt/f/fruitnet-chili-yield-outputs/results/exp01_domain_benchmark
2026-06-22 02:55:10,082 | INFO | Python=3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
2026-06-22 02:55:10,083 | INFO | Platform=Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
2026-06-22 02:55:10,

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      3.14G       1.77       1.71       2.18          7        640: 100% ━━━━━━━━━━━━ 457/457 3.2it/s 2:22<0.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 93/93 3.6it/s 25.9s0.2ss
                   all       1476      44345      0.112      0.214     0.0454     0.0116

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100       3.9G      1.438      1.455      1.912          7        640: 100% ━━━━━━━━━━━━ 457/457 3.8it/s 1:59<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 93/93 4.6it/s 20.4s0.2s
                   all       1476      44345      0.208      0.271      0.109     0.0333

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

     86/100       3.9G     0.8611     0.7216       1.39          7        640: 100% ━━━━━━━━━━━━ 457/457 3.9it/s 1:56<0.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 93/93 4.7it/s 19.7s0.2ss
                   all       1476      44345       0.55      0.381       0.29      0.127

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     87/100       3.9G     0.8584     0.7173      1.388          7        640: 100% ━━━━━━━━━━━━ 457/457 4.0it/s 1:55<0.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 93/93 4.5it/s 20.8s0.2ss
                   all       1476      44345      0.566      0.394      0.299      0.129

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: adaptive_avg_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     88/100       3.9G     0.8588      0.722      1.387          7        640: 100% ━━━━━━━━━━━━ 457/457 4.0it/s 1:55<0.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 93/93 4.8it/s 19.3s0.2s
                   all       1476      44345      0.614      0.411      0.332      0.145
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 58, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

88 epochs completed in 3.448 hours.
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/public_benchmark/gwhd_fruitnet_gcs/weights/last.pt, 28.1MB
Optimizer stripped from /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/public_benchmark/gwhd_fruitnet_gcs/weights/best.pt, 28.1MB

Validating /mnt/f/fruitnet-chili-yield-outputs/runs/detectors/public_b